# HDB Resale Price Impact of Proximity to Good Primary Schools

This notebook uses only files under `cleaned_datasets/` to:
1. Perform informative EDA with useful visualizations.
2. Document methodology for predictive and causal analysis.
3. Train baseline and advanced models.
4. Estimate treatment effect using **Double Machine Learning (DML)** and benchmark against baseline causal models.


## Data Used

- `cleaned_datasets/hdb_resale_mrt.csv`
- `cleaned_datasets/schools_tiered.csv`

Notes:
- `hdb_resale_mrt.csv` contains resale transactions, point geometry, and MRT accessibility features.
- `schools_tiered.csv` contains school quality scores and school tiers.


In [ ]:
# Install dependencies if needed
%pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels scipy


In [ ]:
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


In [ ]:
# File paths
HDB_PATH = Path('cleaned_datasets/hdb_resale_mrt.csv')
SCHOOL_PATH = Path('cleaned_datasets/schools_tiered.csv')

assert HDB_PATH.exists(), f'Missing {HDB_PATH}'
assert SCHOOL_PATH.exists(), f'Missing {SCHOOL_PATH}'

hdb = pd.read_csv(HDB_PATH)
sch = pd.read_csv(SCHOOL_PATH)

print('HDB shape:', hdb.shape)
print('School shape:', sch.shape)
display(hdb.head(2))
display(sch.head(2))


In [ ]:
# -------------------------
# Cleaning and feature engineering
# -------------------------

# Remove unnamed index-like column if present
if '' in hdb.columns:
    hdb = hdb.drop(columns=[''])

# Parse dates and key numeric fields
hdb['month'] = pd.to_datetime(hdb['month'], errors='coerce')
hdb['resale_price'] = pd.to_numeric(hdb['resale_price'], errors='coerce')
hdb['floor_area_sqm'] = pd.to_numeric(hdb['floor_area_sqm'], errors='coerce')
hdb['lease_commence_date'] = pd.to_numeric(hdb['lease_commence_date'], errors='coerce')
hdb['walking_dist_m'] = pd.to_numeric(hdb['walking_dist_m'], errors='coerce')

# Parse geometry: POINT (lon lat)
geom = hdb['geometry'].astype(str).str.extract(r'POINT \(([-0-9\.]+) ([-0-9\.]+)\)')
hdb['lon'] = pd.to_numeric(geom[0], errors='coerce')
hdb['lat'] = pd.to_numeric(geom[1], errors='coerce')

# Calendar and housing age
hdb['year'] = hdb['month'].dt.year
hdb['month_num'] = hdb['month'].dt.month
hdb['flat_age'] = hdb['year'] - hdb['lease_commence_date']
hdb['log_price'] = np.log(hdb['resale_price'])

# Parse storey range into midpoint (e.g., '04 TO 06' -> 5)
storey = hdb['storey_range'].astype(str).str.extract(r'(\d+)\s+TO\s+(\d+)')
hdb['storey_low'] = pd.to_numeric(storey[0], errors='coerce')
hdb['storey_high'] = pd.to_numeric(storey[1], errors='coerce')
hdb['storey_mid'] = (hdb['storey_low'] + hdb['storey_high']) / 2

# Postal sector proxy (first 2 digits) used for neighborhood matching to school quality
hdb['postal_code'] = hdb['postal_code'].astype(str).str.zfill(6)
hdb['postal_sector'] = hdb['postal_code'].str[:2]

# School quality features
sch['postal_code'] = sch['postal_code'].astype(str).str.zfill(6)
sch['postal_sector'] = sch['postal_code'].str[:2]
sch['good_tier12'] = sch['school_tier'].isin(['Tier 1', 'Tier 2']).astype(int)
sch['good_tier1'] = (sch['school_tier'] == 'Tier 1').astype(int)

# Aggregate school quality at postal-sector level (coarse proximity proxy)
sector_school = (
    sch.groupby('postal_sector', as_index=False)
      .agg(
          school_count=('school_name', 'count'),
          good_tier12_count=('good_tier12', 'sum'),
          good_tier1_count=('good_tier1', 'sum'),
          avg_school_score=('school_score', 'mean'),
          max_school_score=('school_score', 'max')
      )
)

# Merge sector school quality into HDB
hdb = hdb.merge(sector_school, on='postal_sector', how='left')
for c in ['school_count', 'good_tier12_count', 'good_tier1_count', 'avg_school_score', 'max_school_score']:
    hdb[c] = hdb[c].fillna(0)

# Treatment definitions
hdb['near_good_school'] = (hdb['good_tier12_count'] > 0).astype(int)
hdb['near_tier1_school'] = (hdb['good_tier1_count'] > 0).astype(int)
hdb['high_score_school_sector'] = (hdb['avg_school_score'] >= hdb['avg_school_score'].quantile(0.75)).astype(int)

# Final analytic subset
hdb = hdb.dropna(subset=['month', 'resale_price', 'floor_area_sqm', 'flat_age', 'log_price'])
hdb = hdb[hdb['resale_price'] > 0].copy()

print('Final modeling table:', hdb.shape)


## EDA 1: Data Quality and Structure


In [ ]:
missing = hdb.isna().sum().sort_values(ascending=False)
display(missing.to_frame('missing_count').head(20))

print('Rows:', len(hdb))
print('Date range:', hdb['month'].min().date(), 'to', hdb['month'].max().date())
print('Unique towns:', hdb['town'].nunique())
print('Unique MRT stations:', hdb['nearest_mrt'].nunique())


## EDA 2: Price Distribution and Time Trend


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(hdb['resale_price'], bins=60, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Resale Price Distribution')
axes[0].set_xlabel('Resale Price (SGD)')

sns.histplot(hdb['log_price'], bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Log Resale Price Distribution')
axes[1].set_xlabel('log(Resale Price)')

plt.tight_layout()
plt.show()


In [ ]:
trend = hdb.groupby('month', as_index=False)['resale_price'].median()

plt.figure(figsize=(12, 5))
plt.plot(trend['month'], trend['resale_price'], color='navy')
plt.title('Median HDB Resale Price Over Time')
plt.xlabel('Month')
plt.ylabel('Median Resale Price (SGD)')
plt.tight_layout()
plt.show()


## EDA 3: Price by Key Market Segments


In [ ]:
# Top towns by volume for readability
top_towns = hdb['town'].value_counts().head(12).index
plot_df = hdb[hdb['town'].isin(top_towns)].copy()

town_stats = plot_df.groupby('town', as_index=False)['resale_price'].median().sort_values('resale_price', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=town_stats, x='resale_price', y='town', palette='viridis')
plt.title('Median Resale Price by Town (Top 12 by Volume)')
plt.xlabel('Median Resale Price (SGD)')
plt.ylabel('Town')
plt.tight_layout()
plt.show()


In [ ]:
flat_order = (hdb.groupby('flat_type')['resale_price'].median().sort_values().index)

plt.figure(figsize=(10, 6))
sns.boxplot(data=hdb, x='flat_type', y='resale_price', order=flat_order, showfliers=False, palette='Set2')
plt.title('Resale Price by Flat Type')
plt.xlabel('Flat Type')
plt.ylabel('Resale Price (SGD)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
# Accessibility signal: walking distance to nearest MRT vs price
sample_df = hdb.sample(min(10000, len(hdb)), random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_df, x='walking_dist_m', y='resale_price', alpha=0.25, s=18)
plt.title('Resale Price vs Walking Distance to MRT')
plt.xlabel('Walking Distance to MRT (m)')
plt.ylabel('Resale Price (SGD)')
plt.tight_layout()
plt.show()


## EDA 4: School Quality Exposure (Sector Proxy)

Important methodological note:
- Since school coordinates are not in `cleaned_datasets/schools_tiered.csv`, this notebook uses **postal sector (first 2 digits)** as a coarse location proxy for school proximity.


In [ ]:
# School tier distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=sch, x='school_tier', order=sch['school_tier'].value_counts().index, palette='mako')
plt.title('Distribution of School Tiers')
plt.xlabel('School Tier')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Treatment prevalence in transaction data
treat_share = hdb['near_good_school'].mean() * 100
print(f'Treatment prevalence (near_good_school=1): {treat_share:.2f}%')

plt.figure(figsize=(6, 4))
sns.countplot(data=hdb, x='near_good_school', palette='coolwarm')
plt.title('Transactions by Near-Good-School Treatment')
plt.xlabel('near_good_school')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Raw outcome gap
raw_gap = hdb.groupby('near_good_school')['resale_price'].median()
display(raw_gap.to_frame('median_resale_price'))

plt.figure(figsize=(7, 5))
sns.boxplot(data=hdb.sample(min(20000, len(hdb)), random_state=42), x='near_good_school', y='resale_price', showfliers=False, palette='Pastel1')
plt.title('Raw Price Difference by Treatment Group')
plt.xlabel('near_good_school')
plt.ylabel('Resale Price (SGD)')
plt.tight_layout()
plt.show()


In [ ]:
# Monthly median by treatment group
grp = (hdb.groupby(['month', 'near_good_school'])['resale_price']
         .median()
         .reset_index())

plt.figure(figsize=(12, 6))
for k, g in grp.groupby('near_good_school'):
    label = 'Near good school' if k == 1 else 'Not near good school'
    plt.plot(g['month'], g['resale_price'], label=label)

plt.title('Median Monthly Resale Price by Treatment Group')
plt.xlabel('Month')
plt.ylabel('Median Resale Price (SGD)')
plt.legend()
plt.tight_layout()
plt.show()


## EDA 5: Correlation Heatmap for Numeric Features


In [ ]:
num_cols = [
    'resale_price', 'log_price', 'floor_area_sqm', 'flat_age', 'storey_mid',
    'walking_dist_m', 'school_count', 'good_tier12_count', 'good_tier1_count',
    'avg_school_score', 'max_school_score', 'near_good_school'
]

corr = hdb[num_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()


## Methodology

### Problem Framing
- Outcome (`Y`): `log_price = log(resale_price)`
- Treatment (`D`): `near_good_school` (1 if postal sector has >=1 Tier1/2 school)
- Controls (`X`): housing attributes, MRT accessibility, town, month fixed effects, and other structural variables.

### Why this setup
- Log price stabilizes variance and allows semi-elastic interpretation.
- Baseline predictive models verify signal quality.
- Causal models estimate treatment effect while adjusting for confounders.

### Modeling Stack
1. **Baselines (predictive)**
   - Global median predictor
   - Hedonic linear regression (with one-hot encoded categorical features)
2. **Advanced predictive models**
   - Random Forest Regressor
   - HistGradientBoosting Regressor
3. **Causal estimators**
   - Fixed-effects OLS (town + month FE)
   - **Double Machine Learning (partially linear model with cross-fitting)**


In [ ]:
# -------------------------
# Train-test split (out-of-time)
# -------------------------
cutoff = hdb['month'].quantile(0.80)
train_df = hdb[hdb['month'] <= cutoff].copy()
test_df = hdb[hdb['month'] > cutoff].copy()

print('Cutoff month:', cutoff.date())
print('Train rows:', len(train_df), 'Test rows:', len(test_df))


In [ ]:
# Features for prediction
feature_num = ['floor_area_sqm', 'flat_age', 'storey_mid', 'walking_dist_m', 'school_count', 'avg_school_score']
feature_cat = ['town', 'flat_type', 'flat_model', 'storey_range', 'nearest_mrt']
feature_bin = ['near_good_school']

X_train = train_df[feature_num + feature_cat + feature_bin]
X_test = test_df[feature_num + feature_cat + feature_bin]

y_train = train_df['log_price']
y_test = test_df['log_price']

# Build two preprocessors:
# - sparse for models that support sparse matrices (Linear/RF)
# - dense for HistGradientBoosting (it does NOT support sparse input)
try:
    oh_sparse = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    oh_dense = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    # Backward compatibility for older sklearn versions
    oh_sparse = OneHotEncoder(handle_unknown='ignore', sparse=True)
    oh_dense = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_sparse = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), feature_num + feature_bin),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('oh', oh_sparse)
    ]), feature_cat)
])

preprocess_dense = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), feature_num + feature_bin),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('oh', oh_dense)
    ]), feature_cat)
])

# Default preprocessor for sparse-compatible models
preprocess = preprocess_sparse



In [ ]:
# Evaluation helper

def eval_metrics(y_true_log, y_pred_log):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, r2, mape


In [ ]:
# -------------------------
# Baseline 1: Global median predictor
# -------------------------
median_log = y_train.median()
pred_median = np.repeat(median_log, len(y_test))

mae, rmse, r2, mape = eval_metrics(y_test.values, pred_median)
results = [{
    'model': 'baseline_global_median',
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'MAPE_pct': mape
}]

pd.DataFrame(results)


In [ ]:
# -------------------------
# Baseline 2: Hedonic linear regression
# -------------------------
lin_pipe = Pipeline([
    ('prep', preprocess),
    ('model', LinearRegression())
])
lin_pipe.fit(X_train, y_train)
pred_lin = lin_pipe.predict(X_test)

mae, rmse, r2, mape = eval_metrics(y_test.values, pred_lin)
results.append({
    'model': 'linear_regression',
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'MAPE_pct': mape
})

pd.DataFrame(results)


In [ ]:
# -------------------------
# Advanced 1: Random Forest
# -------------------------
rf_pipe = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])
rf_pipe.fit(X_train, y_train)
pred_rf = rf_pipe.predict(X_test)

mae, rmse, r2, mape = eval_metrics(y_test.values, pred_rf)
results.append({
    'model': 'random_forest',
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'MAPE_pct': mape
})

pd.DataFrame(results)


In [ ]:
# -------------------------
# Advanced 2: HistGradientBoosting
# -------------------------
# Robust guard: HistGradientBoostingRegressor requires dense input.
# This cell will work even if the upstream preprocessor returns sparse matrices.
from sklearn.preprocessing import FunctionTransformer

prep_for_hgb = preprocess_dense if 'preprocess_dense' in globals() else preprocess

def _to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else X

hgb_pipe = Pipeline([
    ('prep', prep_for_hgb),
    ('to_dense', FunctionTransformer(_to_dense, accept_sparse=True)),
    ('model', HistGradientBoostingRegressor(
        max_depth=10,
        max_iter=450,
        learning_rate=0.05,
        random_state=42
    ))
])
hgb_pipe.fit(X_train, y_train)
pred_hgb = hgb_pipe.predict(X_test)

mae, rmse, r2, mape = eval_metrics(y_test.values, pred_hgb)
results.append({
    'model': 'hist_gradient_boosting',
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'MAPE_pct': mape
})

res_df = pd.DataFrame(results).sort_values('RMSE')
display(res_df)



In [ ]:
# Visualize predictive model comparison
plot_df = res_df.melt(id_vars='model', value_vars=['MAE', 'RMSE', 'MAPE_pct'], var_name='metric', value_name='value')

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x='metric', y='value', hue='model')
plt.title('Predictive Performance Comparison (Lower is Better)')
plt.xlabel('Metric')
plt.ylabel('Value')
plt.tight_layout()
plt.show()


In [ ]:
# Actual vs Predicted (best RMSE model)
best_name = res_df.iloc[0]['model']
if best_name == 'hist_gradient_boosting':
    pred_best = pred_hgb
elif best_name == 'random_forest':
    pred_best = pred_rf
elif best_name == 'linear_regression':
    pred_best = pred_lin
else:
    pred_best = pred_median

actual = np.exp(y_test.values)
predicted = np.exp(pred_best)

plt.figure(figsize=(7, 7))
plt.scatter(actual, predicted, alpha=0.2, s=15)
lim_min = min(actual.min(), predicted.min())
lim_max = max(actual.max(), predicted.max())
plt.plot([lim_min, lim_max], [lim_min, lim_max], color='red', linestyle='--')
plt.title(f'Actual vs Predicted Prices ({best_name})')
plt.xlabel('Actual resale_price')
plt.ylabel('Predicted resale_price')
plt.tight_layout()
plt.show()


## Causal Baseline: Fixed-Effects OLS

Model:
`log_price ~ near_good_school + controls + C(town) + C(month)`

Interpretation:
- `% effect = 100 * (exp(beta_treatment) - 1)`


In [ ]:
causal_df = hdb.copy()
causal_df['month_str'] = causal_df['month'].dt.to_period('M').astype(str)

formula = (
    'log_price ~ near_good_school + floor_area_sqm + flat_age + storey_mid + walking_dist_m '
    '+ C(flat_type) + C(flat_model) + C(storey_range) + C(town) + C(month_str)'
)

fe_ols = smf.ols(formula=formula, data=causal_df).fit(cov_type='HC1')
print(fe_ols.summary().tables[1])

b = fe_ols.params.get('near_good_school', np.nan)
if pd.notna(b):
    print(f'FE OLS estimated treatment effect: {100*(np.exp(b)-1):.2f}%')


## Double Machine Learning (DML): Partially Linear Model

We estimate:
- `Y = theta*D + g(X) + noise`
- `D = m(X) + noise`

Procedure:
1. Cross-fit nuisance models `g(X)` and `m(X)`.
2. Residualize `Y` and `D`.
3. Regress residualized `Y` on residualized `D` to get debiased `theta`.


In [ ]:
# Prepare DML dataset
dml_df = hdb.copy()
dml_df['month_str'] = dml_df['month'].dt.to_period('M').astype(str)

D = dml_df['near_good_school'].astype(float).values
Y = dml_df['log_price'].astype(float).values

X_cols_num = ['floor_area_sqm', 'flat_age', 'storey_mid', 'walking_dist_m', 'school_count', 'avg_school_score']
X_cols_cat = ['town', 'flat_type', 'flat_model', 'storey_range', 'nearest_mrt', 'month_str']

X = dml_df[X_cols_num + X_cols_cat]

x_prep = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), X_cols_num),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore'))
    ]), X_cols_cat)
])

X_tr = x_prep.fit_transform(X)


In [ ]:
# Cross-fitting
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_hat = np.zeros(len(dml_df))
d_hat = np.zeros(len(dml_df))

for tr_idx, te_idx in kf.split(X_tr):
    Xtr, Xte = X_tr[tr_idx], X_tr[te_idx]
    ytr, dtr = Y[tr_idx], D[tr_idx]

    # Outcome model g(X)
    g_model = RandomForestRegressor(
        n_estimators=250,
        max_depth=15,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    g_model.fit(Xtr, ytr)
    y_hat[te_idx] = g_model.predict(Xte)

    # Treatment model m(X)
    m_model = RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    m_model.fit(Xtr, dtr)
    d_hat[te_idx] = m_model.predict_proba(Xte)[:, 1]

# Residual-on-residual estimate
y_tilde = Y - y_hat
d_tilde = D - d_hat

theta = np.sum(d_tilde * y_tilde) / np.sum(d_tilde ** 2)

# Influence-function based standard error
psi = (y_tilde - theta * d_tilde) * d_tilde
n = len(dml_df)
se = np.sqrt(np.mean(psi**2) / (np.mean(d_tilde**2)**2 * n))
ci_low = theta - 1.96 * se
ci_high = theta + 1.96 * se

pct = 100 * (np.exp(theta) - 1)
pct_low = 100 * (np.exp(ci_low) - 1)
pct_high = 100 * (np.exp(ci_high) - 1)

print(f'DML theta (log points): {theta:.4f}')
print(f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'DML estimated effect (%): {pct:.2f}%')
print(f'95% CI in % terms: [{pct_low:.2f}%, {pct_high:.2f}%]')


## Robustness and Evaluation Checks


In [ ]:
# Robustness 1: alternate treatment definitions in FE OLS
rob_rows = []
for tr in ['near_good_school', 'near_tier1_school', 'high_score_school_sector']:
    f = (
        f'log_price ~ {tr} + floor_area_sqm + flat_age + storey_mid + walking_dist_m '
        '+ C(flat_type) + C(flat_model) + C(storey_range) + C(town) + C(month_str)'
    )
    tmp = causal_df.copy()
    m = smf.ols(f, data=tmp).fit(cov_type='HC1')
    b = m.params.get(tr, np.nan)
    se = m.bse.get(tr, np.nan)
    rob_rows.append({
        'treatment': tr,
        'coef_log': b,
        'std_err': se,
        'effect_pct': 100 * (np.exp(b) - 1) if pd.notna(b) else np.nan
    })

rob_df = pd.DataFrame(rob_rows)
display(rob_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=rob_df, x='treatment', y='effect_pct', palette='crest')
plt.title('Robustness: Effect by Treatment Definition')
plt.xlabel('Treatment Definition')
plt.ylabel('Estimated Effect (%)')
plt.tight_layout()
plt.show()


In [ ]:
# Placebo test: random treatment preserving prevalence
np.random.seed(42)
placebo = np.random.binomial(1, causal_df['near_good_school'].mean(), size=len(causal_df))
causal_df['placebo_treat'] = placebo

placebo_model = smf.ols(
    'log_price ~ placebo_treat + floor_area_sqm + flat_age + storey_mid + walking_dist_m + C(flat_type) + C(town) + C(month_str)',
    data=causal_df
).fit(cov_type='HC1')

bp = placebo_model.params.get('placebo_treat', np.nan)
print(placebo_model.summary().tables[1])
print(f'Placebo estimated effect: {100*(np.exp(bp)-1):.2f}%')


In [ ]:
# Consolidated results table
rows = []

# Predictive best model
best_pred = res_df.iloc[0]
rows.append({
    'category': 'Predictive',
    'method': best_pred['model'],
    'main_metric': f"RMSE={best_pred['RMSE']:,.2f}, MAE={best_pred['MAE']:,.2f}, R2={best_pred['R2']:.3f}"
})

# FE OLS
if 'near_good_school' in fe_ols.params.index:
    b = fe_ols.params['near_good_school']
    rows.append({
        'category': 'Causal',
        'method': 'FE OLS',
        'main_metric': f"Effect={100*(np.exp(b)-1):.2f}%"
    })

# DML
rows.append({
    'category': 'Causal',
    'method': 'Double Machine Learning',
    'main_metric': f"Effect={pct:.2f}% (95% CI: {pct_low:.2f}%, {pct_high:.2f}%)"
})

summary = pd.DataFrame(rows)
display(summary)


## Conclusion Template (Fill After Running)

- Baseline predictive performance shows whether school exposure adds practical forecasting value.
- FE OLS gives a first-pass adjusted estimate of school proximity premium.
- DML provides a more robust estimate under flexible nuisance functions.
- Robustness checks (alternate treatments + placebo) help validate whether the estimated premium is stable and credible.

## Key Limitation
- Proximity is approximated via **postal sector**, not exact geodesic distance to school gate. If school coordinates become available, replace treatment with true distance-based thresholds (e.g., <=1km).
